In [0]:
dbutils.widgets.removeAll()

In [0]:
# Set input parameters - point to external JSON config file
import os
try:
    notebook_dir = os.getcwd()
    project_root = os.path.abspath(os.path.join(notebook_dir, "../../.."))
except Exception:
    project_root = "/tmp"

clientContainer = "new"
subGroupConfigPath = os.path.join(project_root, "DimProvider/Gold/Schema/dimProvider.json")

print(f"Client Container: {clientContainer}")
print(f"Config Path: {subGroupConfigPath}")
print(f"Source: {clientContainer}.silver.provider_hierarchy → Destination: {clientContainer}.gold.dimprovider")

In [0]:
# Setup widgets (if not already defined)
dbutils.widgets.text("ClientContainer", "", "Client Container")
dbutils.widgets.text("SubGroupConfigPath", "", "SubGroup Config Path")

# Get widget values, but use pre-defined values if widgets are empty
widget_client = dbutils.widgets.get("ClientContainer")
widget_config = dbutils.widgets.get("SubGroupConfigPath")

# Use widget values if provided, otherwise use pre-defined variables
if widget_client:
    clientContainer = widget_client
if widget_config:
    subGroupConfigPath = widget_config

# Wrap catalog name in backticks for numerical safety (e.g. `274`)
safe_catalog = f"`{clientContainer}`"

print(f"Client Container: {clientContainer}")
print(f"Safe Catalog: {safe_catalog}")
print(f"Config Path: {subGroupConfigPath}")

In [0]:
from pyspark.sql.functions import explode, col
import os

In [0]:
def path_exists(path):
    """Check if a path exists (table, file, or directory)"""
    try:
        if '.' in path and not path.startswith('/'):
            parts = path.split('.')
            if len(parts) == 3:
                return spark.catalog.tableExists(path)
        if path.startswith('/'):
            try:
                dbutils.fs.ls(path)
                return True
            except:
                return False
        return spark.catalog.tableExists(path)
    except:
        return False

In [0]:
print(f"Reading config from: {subGroupConfigPath}")
import json

with open(subGroupConfigPath, 'r') as f:
    config_data = json.load(f)

subLayerProcessing = config_data.get("SubLayerProcessing", [])
print(f"Found {len(subLayerProcessing)} sub-group entities to process")
for entity in subLayerProcessing:
    print(f"  - {entity.get('SubGroupEntity', 'Unknown')}")

In [0]:
processing_results = []

for entity_row in subLayerProcessing:
    subGroupEntity = entity_row.get("SubGroupEntity", "Unknown")
    print(f"\n{'='*60}")
    print(f"Processing Entity: {subGroupEntity}")
    print(f"{'='*60}")
    
    all_sources_valid = True
    destinationTable = entity_row.get("DestinationTable", "").replace("#clientCode", safe_catalog)
    print(f"Destination: {destinationTable}")
    
    dest_exists = path_exists(destinationTable)
    if not dest_exists:
        print(f"WARNING: Destination table {destinationTable} does not exist yet")
    
    source_tables = entity_row.get("SourceTables", [])
    
    for source_row in source_tables:
        entity_name = source_row.get("Entity", "")
        source_table = source_row.get("SourceTable", "").replace("#clientCode", safe_catalog)
        source_format = source_row.get("SourceFormat", "delta")
        
        print(f"  Loading source: {entity_name} from {source_table}")
        
        if path_exists(source_table):
            try:
                if '.' in source_table and not source_table.startswith('/'):
                    df_file = spark.table(source_table)
                else:
                    df_file = spark.read.format(source_format).load(source_table)
                
                df_file.createOrReplaceTempView(entity_name)
                row_count = df_file.count()
                print(f"  {entity_name} temp view created ({row_count} rows)")
            except Exception as e:
                all_sources_valid = False
                print(f"  Failed to load {entity_name}: {str(e)}")
        else:
            all_sources_valid = False
            print(f"  Path for {entity_name} does not exist: {source_table}")
    
    if all_sources_valid:
        try:
            print(f"\n  Executing SQL Script...")
            mainSQLQuery = entity_row.get("SQLScript", "").replace("#clientCode", safe_catalog)
            
            mDF = spark.sql(mainSQLQuery)
            mDF.createOrReplaceTempView("tempSQLScript")
            row_count = mDF.count()
            print(f"  tempSQLScript temp view created with {row_count} rows")
            
            print(f"\n  Checking destination table...")
            if not dest_exists:
                table_parts = destinationTable.split('.')
                if len(table_parts) == 3:
                    catalog, schema, table = table_parts
                    print(f"  Ensuring schema exists: {catalog}.{schema}")
                    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
                
                print(f"  Creating new destination table: {destinationTable}")
                mDF.limit(0).write.format("delta").mode("overwrite").saveAsTable(destinationTable)
            
            spark.table(destinationTable).createOrReplaceTempView("DestinationTable")
            print(f"  DestinationTable temp view created")
            
            print(f"\n  Executing Merge Script...")
            mergeScript = entity_row.get("MergeScript", "").replace(
                "DestinationTable", destinationTable
            )
            spark.sql(mergeScript)
            print(f" Merge completed successfully for {subGroupEntity}")
            
            processing_results.append({
                "entity": subGroupEntity,
                "status": "SUCCESS",
                "destination": destinationTable,
                "error": ""
            })
            
        except Exception as e:
            error_msg = f"Failed to execute SQL/Merge for {subGroupEntity}: {str(e)}"
            print(f"  {error_msg}")
            processing_results.append({
                "entity": subGroupEntity,
                "status": "FAILED",
                "destination": destinationTable,
                "error": str(e)
            })
    else:
        error_msg = "One or more source paths do not exist"
        print(f"  Skipping {subGroupEntity}: {error_msg}")
        processing_results.append({
            "entity": subGroupEntity,
            "status": "SKIPPED",
            "destination": destinationTable,
            "error": error_msg
        })

In [0]:
print(f"\n{'='*60}")
print("Processing Summary")
print(f"{'='*60}")

df_results = spark.createDataFrame(processing_results)
display(df_results)

success_count = len([r for r in processing_results if r["status"] == "SUCCESS"])
failed_count = len([r for r in processing_results if r["status"] == "FAILED"])
skipped_count = len([r for r in processing_results if r["status"] == "SKIPPED"])

return_message = f"Processed {len(processing_results)} entities: {success_count} succeeded, {failed_count} failed, {skipped_count} skipped"
print(f"\n{return_message}")
dbutils.notebook.exit(return_message)